# KDD: Artefactos base (Parte I)

Se construyen, sobre Spark, los tres objetos que alimentan las Partes II–VI: **vectores de características**,
**matrices de interacción** y **grafos**, y se analiza el particionamiento y el plan de ejecución

### Imports

In [1]:
import os, socket, time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import ArrayType, StructType

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.max_colwidth", 80)

### Utilidades

- Constantes

In [2]:
NOTEBOOK = "b_artefactos"

proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent

raw_dir     = proyecto / "data" / "raw"
bronze_dir  = proyecto / "data" / "bronze"
staging_dir = proyecto / "data" / "staging"
final_dir   = proyecto / "data" / "final"
artef_dir   = proyecto / "data" / "artefactos"
tc_path     = proyecto / "data" / "cambios.csv"
anios = [2023, 2024, 2025]

out_dir = proyecto / "01_EDA_KDD" / "temp" / NOTEBOOK
fig_dir, tab_dir, web_dir = out_dir / "figs", out_dir / "tablas", out_dir / "web"
for d in (fig_dir, tab_dir, web_dir):
    d.mkdir(parents=True, exist_ok=True)

def mostrar_fig(nombre):
    plt.tight_layout()
    plt.savefig(fig_dir / f"{nombre}.png", dpi=150, bbox_inches="tight")
    plt.show()

def guardar_tabla(df, nombre):
    sdf = spark.createDataFrame(df) if isinstance(df, pd.DataFrame) else df
    sdf.coalesce(1).write.mode("overwrite").parquet(str(tab_dir / nombre))
    sdf.toPandas().to_json(web_dir / f"{nombre}.json", orient="records", force_ascii=False, date_format="iso", indent=1)
    return len(df) if isinstance(df, pd.DataFrame) else sdf.count()

def leer(directorio, nombre):
    return spark.read.parquet(str(directorio / nombre))

- Spark

In [3]:
spark_master = os.environ.get("spark_master", "local[*]")

builder = (SparkSession.builder
           .appName(NOTEBOOK)
           .master(spark_master)
           .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
           .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "4g"))
           .config("spark.sql.shuffle.partitions", "24")
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.sql.session.timeZone", "America/Lima"))

if spark_master.startswith("spark://"):
    builder = (builder.config("spark.driver.host", socket.gethostname())
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
SHUFFLE = int(spark.conf.get("spark.sql.shuffle.partitions"))

### Verificación del cluster

In [4]:
sc = spark.sparkContext
estado = sc._jsc.sc().getExecutorMemoryStatus()

print(f"Master        : {sc.master}")
print(f"Executors     : {estado.size() - 1}")
print(f"Cores totales : {sc.defaultParallelism}")
print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
print("UI del job    : http://localhost:4040")

Master        : spark://spark-master:7077
Executors     : 0
Cores totales : 2
Memoria driver: 2.0 GB
UI del job    : http://localhost:4040


### Carga

In [5]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

hechos = leer(final_dir, "hechos").where(F.col("anio").isin(anios))
proc   = leer(final_dir, "procesos").where(F.col("anio").isin(anios))
part   = leer(final_dir, "participantes").where(F.col("anio").isin(anios))
dim_e, dim_p = leer(final_dir, "entidades"), leer(final_dir, "proveedores")

# nivel adjudicación-proveedor (deshace la explosión por familia)
hp = (hechos.groupBy("ocid", "award_id", "proveedor_key", "entidad_key", "region", "categoria",
                     "postor_unico", "es_directa", "n_proveedores_adj", "anio", "mes")
            .agg(F.sum("monto_asignado").alias("monto"))).cache()
print(f"adjudicaciones-proveedor: {hp.count():,}")

adjudicaciones-proveedor: 74,507


---
### 1. Vectores de características

#### a. Proveedor

In [6]:
w_p = Window.partitionBy("proveedor_key")
dep_p = (hp.groupBy("proveedor_key", "entidad_key").agg(F.sum("monto").alias("m"))
           .withColumn("s", F.col("m") / F.sum("m").over(w_p))
           .groupBy("proveedor_key").agg(F.max("s").alias("share_top_entidad"), F.sum(F.col("s") ** 2).alias("hhi_entidades")))
fam_p = hechos.where("familia <> 'SIN'").groupBy("proveedor_key").agg(F.countDistinct("familia").alias("n_familias"))
ofe_p = part.groupBy("proveedor_key").agg(F.sum("en_lista_oferentes").alias("n_ofertas"),
                                          F.sum(F.col("en_lista_oferentes") * F.col("gano")).alias("n_ganadas_compitiendo"))
cat = lambda v: F.avg((F.col("categoria") == v).cast("int"))

feat_prov = (hp.groupBy("proveedor_key").agg(
        F.countDistinct("ocid").alias("n_procesos"), F.countDistinct("entidad_key").alias("n_entidades"),
        F.countDistinct("region").alias("n_regiones"), F.sum("monto").alias("monto_total"),
        F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano"),
        F.avg("postor_unico").alias("pct_postor_unico"), F.avg("es_directa").alias("pct_directa"),
        F.avg((F.col("n_proveedores_adj") > 1).cast("int")).alias("pct_consorcio"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"),
        F.countDistinct("anio", "mes").alias("meses_activos"))
    .join(dep_p, "proveedor_key", "left").join(fam_p, "proveedor_key", "left").join(ofe_p, "proveedor_key", "left")
    .withColumn("compite", (F.coalesce("n_ofertas", F.lit(0)) > 0).cast("int"))
    .withColumn("tasa_exito", F.when(F.col("n_ofertas") > 0, F.col("n_ganadas_compitiendo") / F.col("n_ofertas")))
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_total"))
    .withColumn("log_monto_mediano", F.log1p("monto_mediano")))

FEAT_PROV = ["log_n_procesos", "n_entidades", "n_regiones", "log_monto_total", "log_monto_mediano", "pct_postor_unico",
             "pct_directa", "pct_consorcio", "pct_bienes", "pct_servicios", "pct_obras", "meses_activos",
             "share_top_entidad", "hhi_entidades", "n_familias", "compite", "tasa_exito"]
feat_prov = VectorAssembler(inputCols=FEAT_PROV, outputCol="features_raw").transform(feat_prov.fillna(0, FEAT_PROV))

#### b. Entidad

Las métricas de proceso salen de **todos** sus procesos, adjudicados o no

In [7]:
w_e = Window.partitionBy("entidad_key")
dep_e = (hp.groupBy("entidad_key", "proveedor_key").agg(F.sum("monto").alias("m"))
           .withColumn("s", F.col("m") / F.sum("m").over(w_e))
           .groupBy("entidad_key").agg(F.max("s").alias("share_top_proveedor"), F.sum(F.col("s") ** 2).alias("hhi_proveedores"),
                                       F.count("*").alias("n_proveedores")))
adjudicados = hp.select("ocid").distinct().withColumn("adjudicado", F.lit(1))

feat_ent = (proc.join(adjudicados, "ocid", "left").fillna({"adjudicado": 0})
    .groupBy("entidad_key").agg(
        F.count("*").alias("n_procesos"), F.avg("postor_unico").alias("pct_postor_unico"),
        F.avg("es_directa").alias("pct_directa"), F.avg("n_postores").alias("postores_promedio"),
        F.avg(1 - F.col("adjudicado")).alias("pct_sin_adjudicar"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"))
    .join(hp.groupBy("entidad_key").agg(F.sum("monto").alias("monto_adjudicado_total"),
                                        F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano")), "entidad_key", "left")
    .join(dep_e, "entidad_key", "left")
    .join(hechos.where("familia <> 'SIN'").groupBy("entidad_key").agg(F.countDistinct("familia").alias("n_familias")),
          "entidad_key", "left")
    .join(dim_e.select("entidad_key", "region"), "entidad_key", "left")
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_adjudicado_total"))
    .withColumn("log_monto_mediano", F.log1p("monto_mediano")))

FEAT_ENT = ["log_n_procesos", "pct_postor_unico", "pct_directa", "postores_promedio", "pct_sin_adjudicar", "pct_bienes",
            "pct_servicios", "pct_obras", "log_monto_total", "log_monto_mediano", "share_top_proveedor",
            "hhi_proveedores", "n_proveedores", "n_familias"]
feat_ent = VectorAssembler(inputCols=FEAT_ENT, outputCol="features_raw").transform(feat_ent.fillna(0, FEAT_ENT))

- Se guardan las columnas crudas y `features_raw` **sin estandarizar**: estandarización, selección y PCA/SVD se justifican en
  la Parte II con el criterio de k a la vista
- `log1p` en montos y conteos por la cola pesada (P1 §1.c); `compite` separa "nunca oferta" de "tasa 0", que el `fillna(0)`
  igualaría

---
### 2. Matrices de interacción (formato largo/COO, índices enteros para ALS y SVD)

In [8]:
ep = (hp.groupBy("entidad_key", "proveedor_key")
        .agg(F.count("*").alias("n_adj"), F.sum("monto").alias("monto"),
             F.min("anio").alias("anio_min"), F.max("anio").alias("anio_max")))
idx_e = StringIndexer(inputCol="entidad_key", outputCol="entidad_idx").fit(ep)
idx_p = StringIndexer(inputCol="proveedor_key", outputCol="proveedor_idx").fit(ep)
m_ep = (idx_p.transform(idx_e.transform(ep))
          .withColumn("entidad_idx", F.col("entidad_idx").cast("int"))
          .withColumn("proveedor_idx", F.col("proveedor_idx").cast("int"))
          .withColumn("senal_log_monto", F.log1p(F.coalesce("monto", F.lit(0)))))

ef = (hechos.where("familia <> 'SIN'").groupBy("entidad_key", "familia")
            .agg(F.countDistinct("ocid").alias("n_procesos"), F.sum("monto_asignado").alias("monto")))
idx_f = StringIndexer(inputCol="familia", outputCol="familia_idx").fit(ef)
m_ef = (idx_f.transform(idx_e.transform(ef))
          .withColumn("entidad_idx", F.col("entidad_idx").cast("int"))
          .withColumn("familia_idx", F.col("familia_idx").cast("int")))

etiquetas = {"indice_entidad": idx_e.labelsArray[0], "indice_proveedor": idx_p.labelsArray[0],
             "indice_familia": idx_f.labelsArray[0]}
indices = {k: spark.createDataFrame(list(enumerate(v)), ["idx", "clave"]) for k, v in etiquetas.items()}

n_e, n_p, n_f = (len(v) for v in etiquetas.values())
nnz_ep, nnz_ef = m_ep.count(), m_ef.count()
print(f"entidad x proveedor: {n_e:,} x {n_p:,}, nnz {nnz_ep:,}, densidad {nnz_ep / (n_e * n_p):.5%}")
print(f"entidad x familia  : {n_e:,} x {n_f:,}, nnz {nnz_ef:,}, densidad {nnz_ef / (n_e * n_f):.3%}")

entidad x proveedor: 2,833 x 35,638, nnz 58,951, densidad 0.05839%
entidad x familia  : 2,833 x 361, nnz 23,559, densidad 2.304%


- **Señal implícita:** se guardan `n_adj` y `senal_log_monto`; cuál usar como confianza en ALS se decide en la Parte III
- `StringIndexer` asigna índices distribuidos y ordenados por frecuencia (un `row_number()` sin partición llevaría todo a
  una sola tarea)
- `anio_min`/`anio_max` permiten un hold-out temporal sin recalcular la matriz

---
### 3. Grafos

#### a. Bipartito entidad → proveedor

Prefijos `E|` y `P|`: una entidad pública también puede ser proveedor

In [9]:
v_ent = (dim_e.join(ep.select("entidad_key").distinct(), "entidad_key")
              .select(F.concat(F.lit("E|"), "entidad_key").alias("id"), F.lit("entidad").alias("tipo"), "nombre", "region"))
v_prov = (dim_p.join(ep.select("proveedor_key").distinct(), "proveedor_key")
               .select(F.concat(F.lit("P|"), "proveedor_key").alias("id"), F.lit("proveedor").alias("tipo"), "nombre", "region"))
vertices_bip = v_ent.unionByName(v_prov)
aristas_bip = ep.select(F.concat(F.lit("E|"), "entidad_key").alias("src"), F.concat(F.lit("P|"), "proveedor_key").alias("dst"),
                        "n_adj", "monto", "anio_min", "anio_max")

#### b. Co-participación proveedor–proveedor (+ consorcios)

In [10]:
MAX_PART, MIN_COMP = 30, 2          # ajustar con b_exploracion §2 (p99 de oferentes por proceso)

n_part = part.groupBy("ocid").agg(F.count("*").alias("n"))
part_n = (part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
              .repartition(SHUFFLE, "ocid").cache())
pares = (part_n.alias("a").join(part_n.alias("b"),
            (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key")))
         .select(F.col("a.proveedor_key").alias("src"), F.col("b.proveedor_key").alias("dst"), F.col("a.n").alias("n")))
co = pares.groupBy("src", "dst").agg(F.count("*").alias("n_coparticipa"),
                                     F.sum(1.0 / (F.col("n") - 1)).alias("peso_coparticipa"))

cons_n = hp.where("n_proveedores_adj > 1").select("ocid", "award_id", "proveedor_key")
cons = (cons_n.alias("a").join(cons_n.alias("b"), (F.col("a.ocid") == F.col("b.ocid")) &
                               (F.col("a.award_id") == F.col("b.award_id")) &
                               (F.col("a.proveedor_key") < F.col("b.proveedor_key")))
        .groupBy(F.col("a.proveedor_key").alias("src"), F.col("b.proveedor_key").alias("dst"))
        .agg(F.count("*").alias("n_consorcio")))

aristas_co = (co.join(cons, ["src", "dst"], "full").fillna(0, ["n_coparticipa", "peso_coparticipa", "n_consorcio"])
              .where(f"n_coparticipa >= {MIN_COMP} OR n_consorcio >= 1")
              .withColumn("peso", F.col("peso_coparticipa") + F.col("n_consorcio"))
              .withColumn("src", F.concat(F.lit("P|"), "src")).withColumn("dst", F.concat(F.lit("P|"), "dst"))).cache()
n_aristas_co = aristas_co.count()

if n_aristas_co < 1000:   # sin oferentes suficientes -> proyección del bipartito (proveedores que comparten entidades)
    print("aviso: co-participación insuficiente, se usa la proyección por entidades compartidas")
    pe = ep.select("entidad_key", "proveedor_key")
    aristas_co = (pe.alias("a").join(pe.alias("b"), (F.col("a.entidad_key") == F.col("b.entidad_key")) &
                                     (F.col("a.proveedor_key") < F.col("b.proveedor_key")))
                  .groupBy(F.concat(F.lit("P|"), F.col("a.proveedor_key")).alias("src"),
                           F.concat(F.lit("P|"), F.col("b.proveedor_key")).alias("dst"))
                  .agg(F.count("*").alias("peso")).where(f"peso >= {MIN_COMP}")).cache()

extremos = aristas_co.select(F.col("src").alias("id")).unionByName(aristas_co.select(F.col("dst").alias("id")))
vertices_co = v_prov.join(extremos.distinct(), "id")
grado_co = extremos.groupBy("id").count()
print(f"co-participación: {vertices_co.count():,} nodos | {aristas_co.count():,} aristas | grado p50/p90/p99:",
      grado_co.approxQuantile("count", [.5, .9, .99], .01))

co-participación: 16,647 nodos | 664,220 aristas | grado p50/p90/p99: [11.0, 66.0, 5726.0]


- **Criterio de arista:** dos proveedores se conectan si participaron (ofertaron o ganaron) en ≥ 2 procesos con entre 2 y
  `MAX_PART` participantes, **o** si fueron adjudicados juntos en consorcio
- Peso = Σ 1/(n−1) + n_consorcio: compartir un proceso de 3 pesa más que uno de 30; el consorcio es un vínculo explícito
- Tope y umbral evitan las cliques de procesos masivos (b_exploracion §2) y las coincidencias únicas, que son mayormente ruido
- La proyección por entidades compartidas queda como respaldo; el grafo KNN sobre vectores estandarizados se construirá
  en la Parte II/VI

---
### 4. Particionamiento y plan de ejecución

In [11]:
def tiempo(df):
    t0 = time.time(); n = df.count(); return n, round(time.time() - t0, 2)

part_sin = part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
pares_sin = (part_sin.alias("a").join(part_sin.alias("b"),
                (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key"))))

print("--- self-join SIN reparticionar (part leído de disco, particionado por anio/mes) ---")
pares_sin.explain()
print("--- self-join sobre part_n = repartition(ocid) + cache ---")
pares.explain()
print("sin repartition:", tiempo(pares_sin), "| con repartition + cache:", tiempo(pares))

--- self-join SIN reparticionar (part leído de disco, particionado por anio/mes) ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [ocid#74], [ocid#4882], Inner, BuildRight, (proveedor_key#75 < proveedor_key#4883), false
   :- Project [ocid#74, proveedor_key#75, n#2574L]
   :  +- BroadcastHashJoin [ocid#74], [ocid#4872], Inner, BuildRight, false
   :     :- Project [ocid#74, proveedor_key#75]
   :     :  +- Filter (isnotnull(ocid#74) AND isnotnull(proveedor_key#75))
   :     :     +- FileScan parquet [ocid#74,proveedor_key#75,anio#78,mes#79] Batched: true, DataFilters: [isnotnull(ocid#74), isnotnull(proveedor_key#75)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/participantes], PartitionFilters: [anio#78 IN (2023,2024,2025)], PushedFilters: [IsNotNull(ocid), IsNotNull(proveedor_key)], ReadSchema: struct<ocid:string,proveedor_key:string>
   :     +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, stri

In [12]:
print("--- dimensión pequeña: broadcast ---")
hechos.join(F.broadcast(dim_e), "entidad_key").explain()

print("--- partition pruning en hechos ---")
leer(final_dir, "hechos").where("anio = 2025 AND mes = 3").select("ocid").explain()

print("filas por partición en aristas de co-participación (sesgo):")
aristas_co.groupBy(F.spark_partition_id().alias("particion")).count().orderBy(F.desc("count")).show(5)

--- dimensión pequeña: broadcast ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [entidad_key#3, ocid#0, award_id#1, fecha#2, proveedor_key#4, region#5, categoria#6, metodo_detalle#7, postor_unico#8, es_directa#9, n_postores#10, n_proveedores_adj#11L, tiene_contrato#12, monto_adj#13, monto_asignado#14, familia#15, segmento#16, anio#17, mes#18, nombre#88, region#89]
   +- BroadcastHashJoin [entidad_key#3], [entidad_key#87], Inner, BuildRight, false
      :- Filter isnotnull(entidad_key#3)
      :  +- FileScan parquet [ocid#0,award_id#1,fecha#2,entidad_key#3,proveedor_key#4,region#5,categoria#6,metodo_detalle#7,postor_unico#8,es_directa#9,n_postores#10,n_proveedores_adj#11L,tiene_contrato#12,monto_adj#13,monto_asignado#14,familia#15,segmento#16,anio#17,mes#18] Batched: true, DataFilters: [isnotnull(entidad_key#3)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/hechos], PartitionFilters: [anio#17 IN (2023,2024,2025)], Pushed

- **En disco:** hechos, procesos y participantes están particionados por `anio/mes`; un filtro temporal (p. ej. el hold-out
  2025) muestra `PartitionFilters` y solo lee esas carpetas (mismo hallazgo del P1)
- **Self-join de co-participación:** leyendo `part` de disco, el plan muestra `Exchange hashpartitioning(ocid)` en cada lado
  (o un `ReusedExchange`). Reparticionando una vez por `ocid` y cacheando, ambos lados leen `InMemoryTableScan` ya
  co-particionado y el `SortMergeJoin` no necesita otro shuffle
- **Dimensiones:** `BroadcastHashJoin`; se copian a cada executor y la tabla de hechos no se mueve
- **Sesgo:** entidades grandes (ESSALUD, Ejército) concentran filas; AQE (activo en Spark 3.5) divide particiones sesgadas
  en los joins, verificable en `localhost:4040`
- `spark.sql.shuffle.partitions = 24` como en el P1: con 2 workers de 1 core, más particiones solo agregan overhead

---
### 5. Exportación de artefactos (`data/artefactos/`)

In [13]:
artefactos = {
    "features_proveedor": feat_prov, "features_entidad": feat_ent,
    "matriz_entidad_proveedor": m_ep, "matriz_entidad_familia": m_ef,
    **{k: v.coalesce(1) for k, v in indices.items()},
    "grafo_bipartito/vertices": vertices_bip, "grafo_bipartito/aristas": aristas_bip,
    "grafo_coparticipacion/vertices": vertices_co, "grafo_coparticipacion/aristas": aristas_co,
}
for nombre, df in artefactos.items():
    df.write.mode("overwrite").parquet(str(artef_dir / nombre))
print("artefactos:", ", ".join(artefactos))

artefactos: features_proveedor, features_entidad, matriz_entidad_proveedor, matriz_entidad_familia, indice_entidad, indice_proveedor, indice_familia, grafo_bipartito/vertices, grafo_bipartito/aristas, grafo_coparticipacion/vertices, grafo_coparticipacion/aristas


### 6. Escritura de resultados

In [14]:
resumen = pd.DataFrame([
    ("entidades (filas features)", feat_ent.count()), ("proveedores (filas features)", feat_prov.count()),
    ("nnz entidad x proveedor", nnz_ep), ("densidad entidad x proveedor", nnz_ep / (n_e * n_p)),
    ("nnz entidad x familia", nnz_ef), ("vertices bipartito", vertices_bip.count()),
    ("aristas bipartito", aristas_bip.count()), ("vertices coparticipacion", vertices_co.count()),
    ("aristas coparticipacion", aristas_co.count())], columns=["medida", "valor"])
tablas = {"resumen_artefactos": resumen,
          "grado_coparticipacion": grado_co.groupBy(F.col("count").alias("grado")).agg(F.count("*").alias("nodos"))}
escritos = {nombre: guardar_tabla(df, nombre) for nombre, df in tablas.items()}
resumen

,medida,valor
0,entidades (filas features),2995.000000
1,proveedores (filas features),35638.000000
2,nnz entidad x proveedor,58951.000000
3,densidad entidad x proveedor,0.000584
4,nnz entidad x familia,23559.000000
5,vertices bipartito,38471.000000
6,aristas bipartito,58951.000000
7,vertices coparticipacion,16647.000000
8,aristas coparticipacion,664220.000000


#### a. Verificación

In [15]:
for nombre, n in escritos.items():
    leido = spark.read.parquet(str(tab_dir / nombre)).count()
    assert leido == n, f"{nombre}: esperaba {n}, leí {leido}"
    assert (web_dir / f"{nombre}.json").exists(), f"falta web/{nombre}.json"

figs = sorted(p.stem for p in fig_dir.glob("*.png"))
print(f"OK: {len(escritos)} tablas (parquet + json)" + (f" | {len(figs)} figuras: {', '.join(figs)}" if figs else ""))

OK: 2 tablas (parquet + json)


---
### 7. Cerrar sesión

In [16]:
spark.stop()